<a href="https://colab.research.google.com/github/PPRRAATTIIKK/Cyber-Bullying-Prediction/blob/main/cyber_bullying_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import joblib
import os
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

nltk.download('stopwords', quiet=True)
print("✅ Libraries imported successfully!\n")

# ========================== 1. LOAD DATASET ==========================
df = pd.read_csv('/content/Formspring.csv')
df = df.dropna()
df.columns = ['text', 'answer']
df['answer'] = df['answer'].str.strip().str.lower()
df = df[df['answer'].isin(['yes', 'no'])]
df['label'] = df['answer'].map({'no': 0, 'yes': 1})

print(f"Dataset shape: {df.shape}")
print("Class distribution:\n", df['label'].value_counts())
print("\n" + "="*60)

# ========================== 2. TEXT PREPROCESSING ==========================
stemmer = SnowballStemmer("english")
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [stemmer.stem(w) for w in text.split() if w not in stop_words and len(w) > 1]
    return " ".join(tokens)

print("Cleaning text...")
df['cleaned_text'] = df['text'].apply(clean_text)

# ========================== 3. FEATURE EXTRACTION ==========================
print("Creating TF-IDF features...")
vectorizer = TfidfVectorizer(max_features=8000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['label']

# ========================== 4. TRAIN/TEST SPLIT ==========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ========================== 5. TRAIN MODEL WITH BALANCED WEIGHTS ==========================
print("Training Logistic Regression with balanced weights...")

model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
model.fit(X_train, y_train)

# ========================== 6. EVALUATION ==========================
y_pred = model.predict(X_test)

print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['Not Bullying', 'Bullying']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ========================== 7. SAVE MODEL (using joblib - more reliable) ==========================
os.makedirs('api', exist_ok=True)

joblib.dump(model, 'api/model.joblib')
joblib.dump(vectorizer, 'api/vectorizer.joblib')

print("\n✅ Model saved successfully to 'api/' folder!")
print("Files created:")
print("   → api/model.joblib")
print("   → api/vectorizer.joblib")
print("\nYou can now deploy this on Vercel.")

✅ Libraries imported successfully!

Dataset shape: (13147, 3)
Class distribution:
 label
0    12295
1      852
Name: count, dtype: int64

Cleaning text...
Creating TF-IDF features...
Training Logistic Regression with balanced weights...

FINAL MODEL PERFORMANCE
              precision    recall  f1-score   support

Not Bullying       0.98      0.94      0.96      3074
    Bullying       0.46      0.69      0.56       213

    accuracy                           0.93      3287
   macro avg       0.72      0.82      0.76      3287
weighted avg       0.94      0.93      0.93      3287


Confusion Matrix:
[[2902  172]
 [  65  148]]

✅ Model saved successfully to 'api/' folder!
Files created:
   → api/model.joblib
   → api/vectorizer.joblib

You can now deploy this on Vercel.
